# 02. 데이터 파이프라인

**목표**
- 언더샘플링 적용 (클래스당 2,000장)
- train / val / test 3-way 분할
- 커스텀 Dataset 클래스 직접 구현
- DataLoader 구성 및 검증

## 1. 라이브러리 및 경로 설정

In [16]:
import os, random
from pathlib import Path
from collections import defaultdict

import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# 재현성
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

RAW_DIR = Path('..') / 'data' / 'raw'
CLASS_MAP = {'고추': 'pepper', '배추': 'cabbage', '오이': 'cucumber'}
SPLITS = ['Training', 'Validation']
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

print('PyTorch:', torch.__version__)
print('CUDA 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('경로 존재:', RAW_DIR.exists())

PyTorch: 2.6.0+cu124
CUDA 사용 가능: True
GPU: NVIDIA GeForce RTX 3080
경로 존재: True


## 2. 전체 이미지 경로 수집

In [17]:
def get_image_paths(raw_dir, splits, class_map):
    data = defaultdict(list)
    for split in splits:
        split_dir = raw_dir / split
        for folder in split_dir.iterdir():
            if not folder.is_dir() or '[원천]' not in folder.name:
                continue
            for kr_name, en_name in class_map.items():
                if kr_name in folder.name:
                    for f in folder.iterdir():
                        if f.suffix.lower() in IMAGE_EXTS:
                            data[en_name].append(f)
    return data

image_paths = get_image_paths(RAW_DIR, SPLITS, CLASS_MAP)

for cls, paths in image_paths.items():
    print(f'{cls:10s}: {len(paths):,}장')

pepper    : 2,705장
cabbage   : 2,008장
cucumber  : 2,227장


## 3. 언더샘플링 (클래스당 2,000장)

EDA 결론: 클래스 간 최대 26% 불균형 → 클래스당 2,000장으로 균일화

In [18]:
MAX_PER_CLASS = 2000

sampled_paths = {}
for cls, paths in image_paths.items():
    sampled = random.sample(paths, MAX_PER_CLASS)
    sampled_paths[cls] = sampled
    print(f'{cls:10s}: {len(sampled):,}장 (원본 {len(paths):,}장에서 샘플링)')

print(f'\n총 이미지 수: {sum(len(v) for v in sampled_paths.values()):,}장')

pepper    : 2,000장 (원본 2,705장에서 샘플링)
cabbage   : 2,000장 (원본 2,008장에서 샘플링)
cucumber  : 2,000장 (원본 2,227장에서 샘플링)

총 이미지 수: 6,000장


## 4. train / val / test 분할 (8 : 1 : 1)

- train 80% : val 10% : test 10%
- `train_test_split` + `stratify` 사용
  - shuffle 기본 보장, 클래스 비율 자동 유지, random_state로 재현성 명시
- test set은 최종 성능 평가 전까지 미사용

In [19]:
CLASS_NAMES = list(sampled_paths.keys())
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASS_NAMES)}

# (path, label) 전체 리스트 생성
all_samples = []
all_labels  = []
for cls, paths in sampled_paths.items():
    label = CLASS_TO_IDX[cls]
    for p in paths:
        all_samples.append((p, label))
        all_labels.append(label)

# train 80% / temp 20% → stratify로 클래스 비율 유지
train_data, temp_data, _, temp_labels = train_test_split(
    all_samples, all_labels,
    test_size=0.2,
    stratify=all_labels,
    random_state=SEED,
)

# temp 20% → val 10% / test 10%
val_data, test_data = train_test_split(
    temp_data,
    test_size=0.5,
    stratify=temp_labels,
    random_state=SEED,
)

print(f'클래스 인덱스: {CLASS_TO_IDX}')
print(f'\ntrain : {len(train_data):,}장')
print(f'val   : {len(val_data):,}장')
print(f'test  : {len(test_data):,}장')
print(f'합계  : {len(train_data)+len(val_data)+len(test_data):,}장')

클래스 인덱스: {'pepper': 0, 'cabbage': 1, 'cucumber': 2}

train : 4,800장
val   : 600장
test  : 600장
합계  : 6,000장


## 4-1. Train 데이터 정규화 통계 계산

- 목적: train 이미지의 RGB 채널별 픽셀 평균(mean)·표준편차(std) 계산 → Normalize 입력값으로 사용
- 총 1,500장 무작위 샘플링 — 전수 계산 대비 소요시간 단축
- train set만 사용 (val/test 정보가 train에 유입되는 것을 방지)

In [ ]:
SAMPLE_N = 500  # 총 샘플 수 = SAMPLE_N × 클래스 수
sample_paths = random.sample([p for p, _ in train_data], min(SAMPLE_N * len(CLASS_NAMES), len(train_data)))

mean = np.zeros(3)
std  = np.zeros(3)
n = len(sample_paths)

for path in sample_paths:
    img = np.array(Image.open(path).convert('RGB').resize((224, 224))) / 255.0
    mean += img.mean(axis=(0, 1))
    std  += img.std(axis=(0, 1))

mean /= n
std  /= n

MEAN = mean.tolist()
STD  = std.tolist()

print(f'샘플 수 : {n}장 (train {len(train_data)}장 중)')
print(f'MEAN : {[round(v, 4) for v in MEAN]}')
print(f'STD  : {[round(v, 4) for v in STD]}')

## 5. 커스텀 Dataset 클래스

- DataLoader는 인덱스로 샘플을 꺼내(`__getitem__`) 배치를 조립하고, 전체 수(`__len__`)로 반복 횟수를 계산
- `(path, label)` 리스트는 파일 경로만 갖고 있어 DataLoader에 바로 넣을 수 없음
- Dataset이 하는 일: `__getitem__` 안에서 경로 → 이미지 로드 → transform → Tensor 변환을 캡슐화
- `torch.utils.data.Dataset` 상속 후 `__len__` / `__getitem__` 직접 구현
  - `transform`: 외부 주입 → train/val/test 전처리 분리 가능

In [21]:
class CropDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

print('CropDataset 클래스 정의 완료')

CropDataset 클래스 정의 완료


## 6. Transform 정의 및 DataLoader 구성

- 목적: 모델이 실제로 받을 수 있는 배치 생성 — 파이프라인 마지막 단계
  - `(path, label)` 리스트 → Dataset → Transform(Tensor 변환) → DataLoader(배치 단위 전달)
- **Transform 결정**
  - train: Resize → RandomHorizontalFlip → ToTensor → Normalize
    - `RandomHorizontalFlip`: 레이블 불변 기하학적 augmentation
  - val / test: Resize → ToTensor → Normalize (augmentation 미적용 — 평가 일관성 유지)
  - Normalize: 4-1에서 계산한 mean/std 사용

In [22]:
IMG_SIZE = 224
BATCH_SIZE = 32  # 관행적 기본값 — GPU 메모리·학습 안정성 고려, 실험으로 조정 가능

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

# Dataset
train_dataset = CropDataset(train_data, transform=train_transform)
val_dataset   = CropDataset(val_data,   transform=val_test_transform)
test_dataset  = CropDataset(test_data,  transform=val_test_transform)

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'BATCH_SIZE  : {BATCH_SIZE}')
print(f'train       : {len(train_dataset):,}장 → {len(train_loader)}배치')
print(f'val         : {len(val_dataset):,}장 → {len(val_loader)}배치')
print(f'test        : {len(test_dataset):,}장 → {len(test_loader)}배치')

BATCH_SIZE  : 32
train       : 4,800장 → 150배치
val         : 600장 → 19배치
test        : 600장 → 19배치


## 7. DataLoader 검증

배치 1개를 꺼내 shape와 값 범위 확인

In [23]:
images, labels = next(iter(train_loader))

print('=== train 배치 검증 ===')
print(f'images shape : {images.shape}')    # (32, 3, 224, 224)
print(f'labels shape : {labels.shape}')    # (32,)
print(f'images dtype : {images.dtype}')
print(f'pixel 범위   : [{images.min():.3f}, {images.max():.3f}]')
print(f'labels 샘플  : {labels[:10].tolist()}')

CLASS_NAMES_LIST = list(CLASS_TO_IDX.keys())
print(f'라벨→클래스  : {dict(enumerate(CLASS_NAMES_LIST))}')

=== train 배치 검증 ===
images shape : torch.Size([32, 3, 224, 224])
labels shape : torch.Size([32])
images dtype : torch.float32
pixel 범위   : [-2.832, 3.165]
labels 샘플  : [1, 0, 1, 0, 1, 0, 0, 0, 1, 2]
라벨→클래스  : {0: 'pepper', 1: 'cabbage', 2: 'cucumber'}


## 8. 데이터 파이프라인 결론

| 항목 | 결정 | 근거 |
|---|---|---|
| 시드 | SEED = 42 | 언더샘플링·분할 재현성 확보 |
| 언더샘플링 | 클래스당 2,000장 | EDA: 클래스 간 최대 26% 불균형 |
| 분할 비율 | train 80% / val 10% / test 10% | 클래스당 2,000장 규모에서 train 4,800장 확보, val·test 각 600장으로 모니터링·평가 가능 |
| 분할 방식 | train_test_split + stratify | shuffle 기본 보장, 클래스 비율 자동 유지, random_state로 재현성 명시 |
| 채널 | RGB | EDA: cabbage/cucumber 채널 분포 유사, 색상 정보 유지 |
| 리사이즈 | 224 × 224 | EDA: CNN 관행적 표준, 최소 해상도 720×720으로 업스케일 없음 |
| Normalize | train 1,500장 샘플 RGB 채널별 픽셀 mean/std 계산 후 적용 | 학습 데이터 분포 기준으로 픽셀값 정규화; 전수 계산 대비 소요시간 단축 |
| Augmentation | train: RandomHorizontalFlip | 레이블 불변 기하학적 augmentation — val/test 미적용 |
| num_workers | 0 | Windows 환경 multiprocessing 이슈 방지 |